In [44]:
import numpy_financial as npf
import numpy as np
from datetime import datetime as dt
import pandas as pd
from datetime import timedelta as td

def set_decimals(n, d = 2):
    return np.round(n,d)


indx = ["TIR", "Cuartil-1", "Mediana", "Cuartil-3", "Min", "Max", "Promedio", "Desv-Estandar"]  

class Portfolio:
    def __init__(self, sheet_name):
        self.name       = sheet_name
        self.info       = pd.read_excel(f"precios.xlsx",sheet_name=sheet_name) # get_excel_info(sheet_name)
        self.primer_dia = self.info.iloc[-1]
        self.ultimo_dia = self.info.iloc[ 0]
        self.anos       = self.info['ano'].unique().tolist()[::-1]

        inf = self.section_info(self.info)

        # TIR
        self.t_tir      = inf[0]
        # Cuartil 1
        self.t_cuatil1  = inf[1]
        # Mediana
        self.t_mediana  = inf[2]
        # Cuartil 3
        self.t_cuatil3  = inf[3]
        # Min
        self.t_min      = inf[4]
        # Max
        self.t_max      = inf[5]
        # Promedio
        self.t_promedio = inf[6]
        # Desviación estandar
        self.t_desv_std = inf[7]

        self.t_year_info, self.t_months_info = self.info_per_year(self.info, self.anos,months=True)
    

    def __str__(self):
        return self.name
    

    def section_info(self, info):
        po = info[['precio']].to_numpy().squeeze()
            
        tir      = np.round(npf.irr([info.iloc[-1]['precio']*-1,info.iloc[0]['precio']])*100, decimals=2)
        cuatil1  = set_decimals(np.quantile(po, 0.25), 4)
        mediana  = set_decimals(np.quantile(po, 0.50), 4)
        cuatil3  = set_decimals(np.quantile(po, 0.75), 4)
        min      = np.rint(np.min(po))
        max      = np.rint(np.max(po))
        promedio = np.rint(np.average(po))
        desv_std = set_decimals(np.std(po),2)

        return [tir,cuatil1,mediana,cuatil3, min, max, promedio, desv_std ]


    def info_per_month(self, year_info):
        tem_mensual = {}
        months_list =  year_info['mes'].unique().tolist()[::-1]
        for m in months_list:
            month_info = year_info[year_info['mes']==m]
            tem_mensual[m] = dict(zip(indx,self.section_info(month_info)))
        return tem_mensual


    def info_per_year(self, info, year_list, months=False):
        dict_anual   = {}
        dict_mensual = {}
        
        for y in year_list:
            year_info = info[info['ano']==y]
            dict_anual[y] = self.section_info(year_info)
            if months == True:
                dict_mensual[y] = self.info_per_month(year_info)
        
        df_anual  = pd.DataFrame(dict_anual, index=indx)
        df_months = pd.DataFrame(dict_mensual)

        if months == True:
            return df_anual.T, df_months.sort_index().T 
        else:
            return df_anual.T
    

    def inf_between_2_dates(self, date1, date2):
        n1 = self.info.index[RN_info.info['fecha'] == date1][0]
        n2 = self.info.index[RN_info.info['fecha'] == date2][0]

        info_date_filter = self.info.iloc[n2: n1+1]

        return pd.DataFrame({f"{date1}/{date2}" : self.section_info(info_date_filter)}, index=indx)
        
    def info_per_month_from_date(self, original_date, months=1):
        parsed_date = dt.strptime(original_date, '%Y-%m-%d')
        li = []

        li.append(str(parsed_date)[:10])

        for i in range(1, months+1):
            li.append(str(parsed_date + td(days=30*i ))[:10])

        dt_per_month = pd.DataFrame(self.inf_between_2_dates(li[0], li[1]))

        for i in range(1, len(li)-1):
            dt_per_month = dt_per_month.join(self.inf_between_2_dates(li[i], li[i+1]))

        dt_per_month = dt_per_month.T

        return dt_per_month

RN_info = Portfolio('Risky Norris')
# RN_info.inf_between_2_dates('2023-02-23', '2023-08-23')
# RN_info.info
# RN_info.t_year_info
# RN_info.t_months_info
# RN_info.t_months_info.map(lambda x: x[0] if isinstance(x, list) else x) ###########
RN_info.t_months_info.map(lambda x: x.get('TIR') if isinstance(x, dict) else None)



,1,2,3,4,5,6,7,8,9,10,11,12
2018,NaN,0.46,-0.44,0.93,5.75,2.16,0.78,8.77,-3.21,-1.52,-2.33,-6.70
2019,3.26,2.52,4.00,3.52,-2.28,2.01,3.88,1.20,2.46,4.61,12.78,-5.29
2020,6.82,-5.68,-8.25,14.12,1.51,6.24,-2.09,11.19,-2.93,-5.17,12.85,-2.45
2021,4.36,-3.22,-3.83,0.85,0.94,5.07,1.08,3.97,-0.05,7.38,4.08,3.50
2022,-13.11,-3.66,1.35,-3.07,-2.47,4.15,5.25,-4.41,-2.94,2.64,0.98,-9.59
2023,2.74,1.63,1.70,0.29,4.48,4.59,8.39,-1.74,-0.51,-3.71,6.70,6.73
2024,5.34,7.99,2.43,-6.70,1.20,7.07,-0.14,0.01,NaN,NaN,NaN,NaN


In [2]:
fecha1 = '2023-02-23'
fecha2 = ''
val = 40
date1 = dt.strptime(fecha1, '%Y-%m-%d')
print(date1)
print(date1+ td(days=val))
c = val//30
r = val%30
print(c)
print(r)
print()

for i in range(1, c+1):
    date2 = date1+ td(days=30)
    print(f"{str(date1)[0:10]} --- {str(date2)[0:10]}")
    date1 = date1+ td(days=30)
if r != 0:
    print(f"{str(date1)[0:10]} --- {str(date2)[0:10]}")

2023-02-23 00:00:00
2023-04-04 00:00:00
1
10

2023-02-23 --- 2023-03-25
2023-03-25 --- 2023-03-25


In [25]:
fecha1 = '2023-02-23'
fecha2 = ''
val = 40
date1 = dt.strptime(fecha1, '%Y-%m-%d')
print(date1)
print(date1+ td(weeks=4))
print(date1+ td(weeks=8))

months = 4
print()
li = []

li.append(str(date1)[:10])
for i in range(1, months+1):
    li.append(str(date1 + td(weeks=4*i ))[:10])

print(li)
print(len(li))
print()

print()
dt_per_month = pd.DataFrame(RN_info.inf_between_2_dates(li[0], li[1]))

for i in range(1, len(li)-1):
    dt_per_month = dt_per_month.join(RN_info.inf_between_2_dates(li[i], li[i+1]))

dt_per_month = dt_per_month.T

pd.DataFrame(dt_per_month)


2023-02-23 00:00:00
2023-03-23 00:00:00
2023-04-20 00:00:00

['2023-02-23', '2023-03-23', '2023-04-20', '2023-05-18', '2023-06-15']
5




,TIR,Cuartil-1,Mediana,Cuartil-3,Min,Max,Promedio,Desv-Estandar
2023-02-23/2023-03-23,1.29,1729.5370,1752.6243,1781.1337,1684.0,1794.0,1751.0,33.24
2023-03-23/2023-04-20,-0.04,1766.4977,1781.3862,1799.9415,1737.0,1827.0,1783.0,20.84
2023-04-20/2023-05-18,2.80,1757.1896,1767.6336,1780.0379,1746.0,1812.0,1769.0,16.12
2023-05-18/2023-06-15,7.08,1837.7158,1862.5852,1873.5488,1810.0,1940.0,1861.0,35.39


In [4]:
fecha1 = '2023-02-23'
fecha2 = '2023-08-23'
n1 = RN_info.info[RN_info.info['fecha'] == fecha1]['precio']
n2 = RN_info.info[RN_info.info['fecha'] == fecha2]['precio']
# n1 = RN_info.info[RN_info.info['fecha'] == fecha1]
# n2 = RN_info.info[RN_info.info['fecha'] == fecha2]
print(n1)
print(n2)


551    1740.5195
Name: precio, dtype: float64
370    2056.8765
Name: precio, dtype: float64


In [5]:
fecha1 = '2023-02-23'
fecha2 = '2023-08-23'
# n1 = RN_info.info[RN_info['fecha'] == fecha1].tolist()[0]
# n2 = RN_info.info[RN_info['fecha'] == fecha2].tolist()[0]
# print(n1)
# print(n2)

# print(type(RN_info.info))
# <class 'pandas.core.frame.DataFrame'>
# RN_info.info.where(RN_info.info['fecha'] == fecha1, inplace=True)
# RN_info.info

n1 = RN_info.info.index[RN_info.info['fecha'] == fecha1][0]
n2 = RN_info.info.index[RN_info.info['fecha'] == fecha2][0]
# n1 = RN_info.info[RN_info.info['fecha'] == fecha1]
# n2 = RN_info.info[RN_info.info['fecha'] == fecha2]
print(n1)
print(n2)

RN_info.info.iloc[n2: n1+1]



551
370


,Unnamed: 0,fecha,ano,mes,dia,precio,accionistas,activos_totales,activos_neto_totales,acciones_en_circulación
370,371,2023-08-23,2023,8,23,2056.8765,46230,219001144851,1.640057e+11,7.973533e+07
371,372,2023-08-22,2023,8,22,2047.9394,46237,218570509205,1.635609e+11,7.986606e+07
372,373,2023-08-21,2023,8,21,2049.1800,46245,218943596682,1.640602e+11,8.006141e+07
373,374,2023-08-20,2023,8,20,2028.8254,46277,216825878204,1.626454e+11,8.016727e+07
374,375,2023-08-19,2023,8,19,2028.8915,46277,216825878204,1.626507e+11,8.016727e+07
...,...,...,...,...,...,...,...,...,...,...
547,548,2023-02-27,2023,2,27,1786.8662,50063,204675394886,1.584049e+11,8.864957e+07
548,549,2023-02-26,2023,2,26,1750.4888,50102,200712217782,1.554114e+11,8.878174e+07
549,550,2023-02-25,2023,2,25,1750.5459,50102,200712217782,1.554165e+11,8.878174e+07
550,551,2023-02-24,2023,2,24,1750.6029,50102,200712217782,1.554216e+11,8.878174e+07
